# ✂️ CutPreacher — cortes de pregação para Reels e Shorts

Gera automaticamente **cortes verticais (9:16) de até 1 minuto, já legendados**, a partir de uma pregação do YouTube.
Uma IA (Claude) lê a pregação inteira e escolhe os momentos mais fortes.

### Como usar (toda semana)
1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4** (deixa tudo ~5x mais rápido). Só precisa fazer uma vez.
2. Clique no ▶ da etapa **1️⃣ Preparar**. Na primeira vez o Google vai pedir permissão para acessar o seu Drive — aceite.
3. Na etapa **2️⃣ Gerar os cortes**, cole o link do YouTube e clique no ▶.
4. Os vídeos ficam no seu Google Drive, na pasta **CutPreacher/saida**. Abra o app do Google Drive no celular para baixar e postar.
   O arquivo `textos_para_postar.txt` traz um título, uma legenda e hashtags para cada corte.

### Configuração (só uma vez)
A IA precisa de uma chave da API da Anthropic:
1. Crie a chave em https://platform.claude.com/settings/keys (é cobrado por uso; cada pregação custa centavos de dólar).
2. Aqui no Colab, clique no ícone de **chave 🔑 (Secrets)** na barra da esquerda → **Adicionar novo secret**.
3. Nome: `ANTHROPIC_API_KEY` — Valor: a sua chave. Ative a opção **Acesso do notebook**.

### Se o YouTube bloquear o download
Às vezes o YouTube bloqueia downloads vindos dos servidores do Google. Nesse caso:
baixe a pregação pelo **YouTube Studio** do canal, envie o arquivo para a pasta **CutPreacher/entrada** do Drive
e preencha o campo `arquivo_no_drive` com o nome do arquivo (ex: `culto-domingo.mp4`).

In [ ]:
#@title 1️⃣ Preparar o sistema (clique no ▶ — uma vez por sessão)
import base64, io, os, subprocess, sys, urllib.request, zipfile
from pathlib import Path

print("Instalando dependências (1-2 min)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "faster-whisper", "yt-dlp[default]", "anthropic"], check=True)

PACKAGE_ZIP = "UEsDBBQAAAAIAAAAIVyZcbCpbQAAAG0AAAAXAAAAY3V0cHJlYWNoZXIvX19pbml0X18ucHlTUlJyLi0JKEpNTM5ILbJSSE8tSlRIzi8qSS1WKEstKslMTswsVtCwtDI001TISU1PzUtJTMkvVkhJVSgoSk1PPLz88Fag0pR8hcj80pDSpFQ9JSUlLq74eKDm4sz8vPh4BVsFJUM9Az0DJS4AUEsDBBQAAAAIAAAAIVwAV15OVgIAADkFAAAXAAAAY3V0cHJlYWNoZXIvX19tYWluX18ucHmFVM2KFDEQvvdTFH3qgelGWRRcaEEQvKgsKIiIhNok3RO2k8rmZx0fRzx48inmxUwn3TMj6Gwu3clX9X2VSlUpbckFQDdadF5WgyMNFsNuUregCniTtlVBOk5mUOOKfJAhKDP6BbTKykkZucIumqqqhBxAozLNBtqX8J6MvK4grSzooD+Kd6/cGLU04SYjjXU09jWPwTqJfCddvQUhPXfKBkWmr99Ih8CTkvTw4vrpc5jkKI1AQT4ZQtQIyXXEw6/DT+rqzZlsh0IwXPSa2lN0XCb+nZxsX79V5g4EwWeKH+OtBIrAUSuzo8KbQr6P6iHvHg6/haRL5C0l4rptKd0khnyJAeMU+tqjEnhUvUEfcKb0mDjxIqUplOXy6T98t7JXJpzIry65txr3q9cwEZ75PXvRPVkjeh1dSR7ow4+90jk6jmLJOjR+czHKVpr7iMLhvJ+zwHekuPT9l9qRzyc1T5DLf+ltB0rsVH89y1ExvKjybae8XcqjeM1Vtl7iHQk5EXwqRtDI/TV4jdO0BS2FinoLU2KT7cNVG6K7pUfuxInuVE76v9QWtAv7cF5BDVmeahanR8gHMuG/3KVAeGq1YgZdCMPMhIBr7S/0idOn1lpU8mfW8c2myrhfOjfZrE3cZGBeJmrGJ2V9P7t0pci2RzjVToaZl2kciMUqnZ5MBje3y1iQv4rgZLO8GtPz+xTL5ehks2STDWqSazD55EwqpcIzoVzBS2YKXFKRplCTodLk2zzPyknpyM32mI6UnUoNwJhJ0TIGfQ81Y/P0YqwuY6uMsuoPUEsDBBQAAAAIAAAAIVzgGukdNwUAAHkMAAAXAAAAY3V0cHJlYWNoZXIvY2FwdGlvbnMucHmNVutu2zYU/q+n4IShkxpGkJve4NUZ3CZpiiVp4WS9wDFUTqJsrhJlUFSczBDQF9nfPVifZIcXXRx7RZmLeA4Pv3POR/KQqShyFEVpJStBowixfFkIiQjnhSSSFbx0HKsT1HFSZR7kRUKzsrH9UIjEcc7HH6MPbydHl2iEDrT06nQ8UdLguRZfj9+BEAZPndM3r0/P4O8KZOGur+MHp2F4/OTk5EHtOhdvJ+fjs27kRDc18nJyPP49Gp9cHU/UMA3iIl+yjHrCnQY//fbt67/41+Fs+su1++3rP/7s4c+u7zhOQlOUFiInMpIsp15J44In5RClWUGkj/YPUSnF0EHQ4hKAc3LrhRgxLj1RVDxpZqCHaBCGvu9r0wWGCHIwT9gN0OHFJUYHT0NoZjy/Nw4CRk/bYbDWzvqjg2ZQUFgMjlJ3vaiH63wYPkrgW+pvsI5Nx7W5RXFGCfckvZVDlclmRhZKjQaCLjMSU8+9vnYxcl2/06yVwutraqXxQVMtl1R4DZNxxpbRCha89PT/IcpYKadqC8wwuCVCWmYxojzps9wZmtBc11USbCNesoSiqZ6tpwHUiskFUgtWQg4Z7MQbimSBPmujzwFM1iBFJYHF6UwLsMxoBXDIhKZ1qrEUrQKARS9GJkSkDAPTPRzpQFtjvQ8KLhmvaKsENwEBImA3qJg9PXWkdoqFsblDoqhLY5Qz7mnPWuwNqvUY2ZVbBUpq9pVdMHBoGZ/DJvxfyjtiO52lV89rjHujHV1xJQTlso/3Q1w28zYpWxChNnRZ5V5GuXdrskJ7aKCxbhWWnam0ysZmvoED+GqotYTl6SoL4Bg/h6grML213G88TPcHM73ixhBKz2a0HUHNsjYOt8zsQMdMT9lMXvl9fnp1KigpEfGiyXQziB8IYMv5Lvq/g2N3k7GwG+rPimVJRMqdJ3jFErkYquKH0YKy+UJaIYUjEXGS0x1FRo+V7G+qyrIumRoGCmYYhM9tPSRiznh005oYdG3zyNosKEmoAIsUjvf0MhZsKdEbnhYzxwhXd0vwf/M4CMM9511G7ia0/DhEa+2ubjSfQGPQa+eDIMtLeZfBtBBASEaTl5ApFWOeXC5IUqyG6I4CNdP3j/eQtixnzom+L4boAvLF6ATS421P5YnRO8Ego7tXRVZUAqNLfUX0FG8rmTFOG/Elib+0/SKDevBGkozFGP3BIRhlCiBSsC8UZkJXRfrRfj/Bd0lixucYjfk8owpD5aDDbX2BlU4IjDI25zlVy3auaT9rOpOm8x6jYx4XCYA6lp8jmpIqk3jdLnVt+yrlGqtL2lzGuqtarxuG0H3edPcHONQ/cKfpP93Hz/ABfoSfhep33eyIGg+A/uMbiLdH/Rm5o4pYUy+PVQm16ZpF+V5iaUpjmHQFR85pLgrFUNmdI3MiQN4qr35b/uYMm2FVuiivciqIpJ6Z2jvLHPxEpgBZvHI6Z6ryzabhzJYmOLhGh17oAmdRELyjKLooON08zpEqXhatLWZ76qw86ZeanmtWIniyaagdZcbiqQuplXFvun8Pbf+e60NwfWCC7axaN/qyALYUgTvIulf36M0mWxskrRgawSvRuFJ4waYrC9CnZ2W43sC4R7TluU39/pVjEeF90AS3fV9sPQpUU2UdAnGRG/xVALlbs+AR1z5367W5B+q1eeTWrvKtE4aQTcbaQBOq+NxF5oaLTUlv8uYi2BHKESNZMa9UMVRHu3sUN1n79ZZePVxq3BQHe6zxWufh7ojFXjm2lu8h95pbbnR0vlU5/wFQSwMEFAAAAAgAAAAhXKpW1OVEAQAAegIAABUAAABjdXRwcmVhY2hlci9jb25maWcucHl1kMFOAyEQhu/7FBO8WlNrmtgmJsbEo158AEKWYZd0YVYGYg8+vLNsu+3BcoDw/9/AzO8SBdDalVwSag0+jJQymBgpm+wpctO4ibEmm3YwzMhnaJGapnm9XOoOX5izjx3vG5AVS9Dt4Efeg48ZXuCpysHHKmvGlqIV1w1kJn+zfljPiDneQLa7EzKY2BXT4R44JzHUmFU17kAl4kwKGLuCQFCvYAnGhJ2xlO5BtRhzEqaVmQxEgoCeRLfIjlqBFAQpS0bqv4uxiaYZ0MvZSjCuRHlvgevHLhkZrVsamruo1k/vecSkA1kcZuAXPimicNNRIYmwWLxm5JFZXNFYeLVdbdU1ic5J9wva+64/fedt7s+hP66f58R6FCAv8m4zy45i1tGES5IfojCmJIG/HyWCNxqsWljW1qebMxAdPLJ2fsD/mD9QSwMEFAAAAAgAAAAhXA0X7QxYBAAAJwkAABcAAABjdXRwcmVhY2hlci9kb3dubG9hZC5weYVVzW7jNhC++ykGLIpIW1fOFnso3KaLNAkWLRbZRX4OQeJqKWlks6FILUnZDpI8Tc/7FHmxDik5khMXzcEROT/fzPCbmdLoCtK0bFxjME1BVLU2DrhS2nEntLKjUXdncFR67Zq7hRTZRvUzHUejq0+XF5e/n6RnJ3BAmkmuq1pIjAz7a+Fcbd9PJ5NotVrdJA/VTRK/j+40QWZ44zUfwuEmyTCesLE3/+PD6aezk6PD85N4NBrlklsLx3qlpObFiTHaRGeNcqLCcIinI6C/mrRIu8AShE07gLQxMrK6MTlOwToTw4+/Qaa1bG0MUuIqXER9DknFXb7ozBKyEnUUx3HnvEQSpktRoB56HsNKm9u0EGYaijKGXOtbgTYtqRJBBR7gVCukEvl/IRTX1BKvW33SmLVhMcbOQmQWogCU1kHDCScxTuBLi/uFEgUOV7q5oFTh8uwjaEMXUudcTo6NWCIE8/BoCXkN3ltjimI7wSATJdDT7y5gV+e21m5BDnzcG2GC65qrorFoOl8DfwFftKWIBn7CG3Bh8cXzluzQfG3EUoN6+kcDqlwrZ3ihp3Df4j2yHqR7xbZGAco6rEZtQi1N71xayLq92rxTUt3Sb1Rzg8rZgwvT4BhwLaxL9W04thC6dpaSvX/GY6U2RBE2BZYt31wvUMwX7teDt/s/78+uce0Oqvrd7IeMt9/v+GzySo2kk2z7apKxcQ9RoZljSm9QNy7t8cjzUIvkrqolC/SKNonBhCRGzIXiMvk+oihiy+KBmdK15HeSMiXLkHcvo7Ljjmul0xU3Sqi5fS3USzQrIxy+ln33TE+D5NqgZ+yf52DaBgZKDQTV14q54n4KQb7gUqKao/0FjrTkGdiFqC3xqMCkx/zbpp0PD3rPClTafzyOfbAFhu/HNpDHDbeHLUlTrmgZPLyNd9HUM+CatWpexmbEh6FVSyxn7nqblaAWaWmXXLW9dPwx8p5i4BbuCrndBkKVmrzSPXUScT13qb/q2msMRdcgA2Lu6kZvXxv0pA6RKV5h5B3FceJDSm1TlmIdscRTqfWD6xzrTY8kjRPSJlv96AMmpT7gXT27FRI79X1bagG1tvbp2xIlZFysuQENy6dvfigVesONBM6R7pH8QEXNTouHFPfOiRNUF3Caiq1KYao9YNswuoG9TLu9MTw7g0zqrw2SRAMNIxqAlABhfdB6LgnqUDo0xDWx5HYK0duYnAcL0ufd1HmB0j114tYOFPebxnE4atxngzxfYHAfxi2F0UD0UxxyxT7VGmUf37lrCqHHL0FQLQWSb8NhB8QEwwDkz1CAgTKm8jA5r4RahJJ2KSQ3att/yY7RcbmgVXRPb/nYS2MIi50uR8Md8F8zOyxHtGF/GIfF89hJ5lJnUT953rD41SLojP9/A7DNEei1KDmqa0Us7J+owE1x1YZq/ZpIBtuh644O+Xp/Ntz87c7whUzm6CIWNiyL/R5l1EZznnPNRv8CUEsDBBQAAAAIAAAAIVygb2SRGQwAAEQfAAAZAAAAY3V0cHJlYWNoZXIvaGlnaGxpZ2h0cy5weZVZX4/bxhF/16fYMgVK1jxGZ9cpoFgODs4FMZDYRu7aIlAEYUWuTnsmuczu8hz7ckCAAv0A7XMfggANWsBPQV/6qm/iT9Lf7JIUqZNs52DfUbvDmdmZ3/xbrbQq2GKxqm2txWLBZFEpbRkvS2W5lao0o1GzdmlU2T5rMVrRm/ZlJcuL9q1HPM/5Mhcjv5mkqlzJbvdMWAti02wWKhO56V7NZRWzc81Lk2pZ2Zj9RelsNDr7+uz89MvFs6+efvnsnE1ZEAR/VunmP2zzb1YXTGTSKs0ywa42rzOhDBOmEqnkuTSWM1FA00wYZhTWpCFCeaHFJTcMYozd/GSS0VnNmeVarDhxzYVm+Nxosvl585Oi1+qCs0qLC+5XBASlKl+DGEKtFukaf1McrOASGlXKipL0YBXXnF1JDZVecT0CiWJfCTp6ptjjEmpeaI6TsLM1DOFWv1b1eb0UyWj0p4ItwTPFjpiMjtiKv2JGlFaCyqhXslyr2Ev4tsZhy7pMSVjNBsqGpT9EJUqcBP/gGCu+swp+tkJD3+hjMIdqYvMzZysnzbGQhYDmbKW5Eexy8yNxJR6ccZyv4V4JfVGXlseMr6QuGqFwLE8tCYjZmkz9i5YgIdHgZoUTCemFLLkznGLQz/ACx1NEluY1fErau1OVikGZzhdeJVUzmQnJPTMN69BeQXwuYI80h2kmDiilkeCw3Lxe5jKF0YiOV3hu1K305keLT3GPO0SJlQDCaJEVyqsmCoBJlXQykk9EwBzwAtuRGuJKAnr8ShplYqZWQluOh7QunD3BgpY1yf0voJmr8gJwNKQuuWXzv4x8KnmGVUn+KfEgOozB0a0rC4fngo4KrDwFDgWiiVB+5bhJIDCT2lt3u7uL7lCWm9dpz7Q8R+iwlSz6K1EyeqozUQrmgA6IOKwWIl8T3gmDsCKwlLCnHYEDXqVIVdjTqKUWCPcEUTwaPfri8bOzxdmjz0+/PEFkX48YfgJkFBFMWKCWlyK1QexXK61gYiuFwZ6ndOsp0sZwacCEa81fNjy6TQk73H7nsPABxQFFBjQIaW0XRsCdmSPr+JZ1sRQ6iFmApOTyHIBEG49bDyhvuZg5iyGsMmWCm9uKOEHAwK8T8xm59FeJsNLmYsDcWI0svof5+ea1rXNwrzVFCUvXvEAJuQLAuEVi/WjMUqAkBaKFiQ5KXCv1/P0EnmzDlOO8qGAuDe7JUQeFacGNY/Ye4p4B5sReGNNGI9ULly0PChjy2A8Yb+gd8Ycpd9T6QlzguLyLQGUsC+/i9Pe8eZBsKAAp1VLqupTRAeYHHcLNGkVqCLEmsnrRtHuCmz389izBBd/WElUar852QiceQjxu4Rg3KIk7B+54K+5pPd8jk2foGyiH58/68fwZz40Ykvc0vhn1FoZq+zTUSHoH9xs0NadPzk+fPDpdnD75FJlPC3RKRSVzEepglvzmkzc//Gs++903wZsf/hnNf//bIBqNRihlxrDP5cU6x397qrXS4Vcouygp7kM0cdIrkIE8EytCJgJwYbumKtw+Tnq9VsSOHjJ4zTPQAoFUsuCbMkguUULCVTC7NonzzCQ5Xt2wN3/7O8MKXOM+z+kDtRM3AYlkVLXYVlKCHENVD8sr5gmjRsFlLfNsURuhF8irxUEFY+Qp3ztOui6SOgmcfkIlEo3HlZLZhFHXN7N1lYvZKlccG+7PfD48IkIF6kzZrPPtKvi01k0nYNH55lQltx0U4N07UAZS8i/7EDnNmwRNTI26nvRCaxWcug6Rs2un6U1TExO0yRn3H1yl9pGJrkjAkM3pEjBcEKpa8E+SMcQIFvT496j5d3uo2+SOAp8Jnk/YvTHSwv373UbU6jt3v+Efb8dOBs58IchSFFEtGq65O7N588M/rpf+0TsevdOSfO+YRB0TZ+2EV9SyhGQVagxcY3PFc2hHbSVeRw9cG07qoqkyroMsrzY/5mh1zcfsyeavT6nxQP9A3VVJtct3E+UaHd611/QmCaLRLZnB+bDdafqgXvmLJt+UD1oPp1w9BPjZnbcGUIR9xMiDD/uvNdK3IdQFkVOoxX2KOQn+4nUmQrhNlAcQ3ouMCaHXodhhPJOpnXs/vZB2zTyXZCksTwoUKLSEFLLIjkXYOcJNXNMONF6BhVvdwpagZNVz9MvT44/G4/F2h5ibKXI0HCf0kQGojtAY5kuePj+6O7770dH4j0fj437CbbfNFOl5xeu831Op2la1XfghcbrTwYkV1VVkzV11/cZOS+cdNahCNK0uTLoWBSfwNk8T1u86+8l9+2heAqHFdDB59uzTWHc6uw60cr1RQH4iIa5/L0mPnuduGntEjLp855LJLjdEmN9JLoRdrDAR5YtmKwRomuBsVuBXVS184WPTKZWiFSInD/pxK9Gj7BQL9EuPT4DNtDaKBkSjMP0gmJr5h9oHQUPzYDhoA+ot4reIeQ8N0PpRh8Ipv0KblZIuE1JGDHNZSJcTmWcXtcKpZsBEJf6E4dKVEJdwXLZp1WqMT5qCBChwyhFtEMXsiSpFd5JSWcfzvQzmRhhEtNIlzFaKcl03A3kyDHdCXIJyk5nQFbmuLWiiflEKjuPbhUR3+l2I5FcL09QsX6ZiuoYAAia+bLlwR6Ic1GUUhtAluzAXZcMkwgGfi5fTnBdLGFJidl+aZm8m5+yo4Ru1CciUvILPFi+UhrY+jdBjow1dvUAZX/JZU0mp3LfPVJ3aUtOtAQbDtZE7wJ5yzL53/vDnwix4AqOXxl04KKBDZL7wAAZOLbZE/cy4RhvFeJlBEzg/xQe7BlraYuzQYxIfLU8pQ9Fk3ZYwWB7AlkUB5pSRiDe9jq7KuhsVUaaCzohVjhUr8xwTMDFslOyDxxkLM7x74cG0sdQWTd5VdEa3ZiTQu+P+2QvfUjkgvyAgO6at2T2yxP4XSeqt17DYARzvPYDU/Rq5Rd8tOAWmnsXMyLlXyRE4c2KvR3in72OAapz8oRXoOQhwIN0e+re34mFISuLUcl06zS9Jcw9jI2PS9w47jracLj0jmNZxmnecGgc0DCeDMrBrdmf6xrULQTr3Wz766SnTqggB/fYc3SvX6Trs1HKh7WA40PSob6iH036EdBK3p3B+Hag2OzqeuwZ5oK/AxNBqRhQ935HQ6dDurTO6/YFSDwZRexCrFaYXkXXQgMfDcTKO+5zI88f3oz6514ZyUyf8Dqju3Y+HDAcQGmRPTTEe9oljdg95bbAOvrTaJlSFKM95ZUJ0gHvSDNrRvesuLS2VygeJlc/GcxhpOYMjyL1L/5nPjtsEbkQuUusa7SZpHhhXfNDfbuha78HHUzK4X8jVxaS7Op/N0AjMfb2aw6SVpvnG0ZHluKXm1Ro39mD7XjzaNoV0h940hR4FrhyaXq51W+0Nv10j28p01O00b0y3e8lJ+9S2IT7zTfvznVu6lR3fXVuHl5BIcJgDrvgryT/2NXdNcwFdRbrWxHUJmkquY8zTVFRWtDOfOzqF97xxKd3fibdNhFtiN7p4u26z0nE8sLdLT722TRrj81nXnpZ14ZGB8KDK3CoY9VNX+x7y2niYuzDK8OfdipugSMMwTZpYSCnCI6ds6oashv8cqrXH7d4HpDBpoXnhJX0TAjirnW8EqFWiyzlMp80Zb6IkaTsaF9muFvOcvP22gSV+6xjfp2tPf4fdbSb2KNqij05W0ck6wUMLwXwDu1KWvW394Tu3LesB+QJnCh0Qwmr30mkeNRihrf710zyKBmyoiaoEuWnYTrnfMUlBBvF/j+dbO9wa7ftbO3N8tGuBVmYT1dSC8PJluM2EDUHMdpGzBzrRbWO1SGpHZygf3SKiTluWtRhstEzbN2+9RSF6e9UZkvScNpqTzfZSgWlHc3yAxl0PTuE3f09I/QxAGEb7qekOkYjdXeI7aP28Q9TNjeM76Hu3kfRS/3LyXVo1F5fT2Zo8vvZuNDTmh8EHQdQ0BcEH1+vmwm1NTg1ty9atuVRGZ+uuQV13td5juKGDt+PmMNoevDXYfMI5oe+8MLJvfnGDHaLYbF5fiRy+81/7aXY94NreibH2ooe+TMzwMOldhXXibqLeNEqFpisC76g1T3pDWyvKadjqlamEnfTzI30PTN/LMcVMvcKYTG3ZJ8N5r5U++j9QSwMEFAAAAAgAAAAhXNe3O1u4AQAAtQQAABUAAABjdXRwcmVhY2hlci9tb2RlbHMucHmVU8tugzAQvPsrLE4Qpf2ASFSV+gmt1ANCyAKbWHVsZC9K8/f12tjQhkqtD4F9MDuenQhrLrTrxAyz5V1H5WUyFijT2gADabQjRGDPwID1ijnHXW5yg+zhuJaOVEiuBkLIc86R8EvfjR1OhPrjgFk4UaEMg5DgetiGwD992YHdg3nl44Vr+C8SRlfPwJ2okg4aZNPSOtItBy7YrKATrAdjbzW2VHvT3yzTrrdyWggopseZjXwdMsw2qLal4SLnNHu5QktC8XmyZuIWbvFzLiLP0nElKvrwtOEbZ+Kx3G9L0+ZKhbHU70NT7H9Mk0L6GtKPAW4ZhvBgOtzaOgCjO+i42ti0EA0SXDiczZDB0BkRrlcuGuEUAANysepVrBMyyZo2OYlnEab8lsx7rl1ThJeiPd51+MVj3T/2qugDLONzrx4kqoPI5eFwrbbyjRzKIjQUR9q01Y/Pq29R3gYK4ekuNy3a3NX+FNrrViYb1fGzFHqq2U9LKYVYSuh1etm17IuS0x//LRLUxslnYz7WyHLm0NXZ5zzuVW6TZ+bOwMZkdJ/+1eTpIqsNA5E7HwZbe6L0YXE4XoJ8AVBLAwQUAAAACAAAACFcXYW2c6AGAACWDwAAFwAAAGN1dHByZWFjaGVyL3BpcGVsaW5lLnB5lVfNjtxEEL7PU7T6Elvxemc3q4hYGkQUCEKgsIKcGAarx27PdNZ2O93t/ckwEg/BkQvigDjwFPsmeRKq+sf2zCYC9jA7ri7XX1d9X02lZEPyvOpNr3ieE9F0UhnC2lYaZoRs9WzmZVumt7VYh8c3Wrbhu+LhmxHN8L1vRSFLXjLDZhX6wW9FzbTmenCkS1GYZDwaNDmaCmr47E46ZjCKcHAJj+7A3HWi3QT5C1bXbF3zmTtMC9bZbML5uhd1mYO/hBS16PIbqUoddGVbicHS99wYMBwOS3nT1pKV4bjiptjm16Lk0mtsxQYi3GzN4EzzmhcmR0fBTAOFqQeFF3DiD4xirS6UWA+5jxKvYp0NleEGbeuOsyuu8tuE8Ft4A0SsL4VMSKfkmidwRW0J5xjDbDb7bKy3/STfcd3XJpsR+JO96XqTl0Jlrr4otMFnpBbaLDHclZVWouZBiqorsF3yiui634jqLjIQTEa0UQlp2G1e8zYjojVkQS7mMTn5FI+cU9QE8aRn0laqhtXiHY/oq5dff04TqxSnvEWViDJdCAFSKjagymmcltyexFOLiqe6X0eKLn9iJ++en/wwP3m2eoyvnQwWIQrRRSCI01recOUtKA5T0VqdZebjX6Vq1CZSEWqvg/rE86oxkYYw2hLqUkGnmMM8m4RoiKoU19ACEdQiaMcJeTo/8FvRXZPNz8t9ttP2f3Byo4TheSe1yTE2HeFQuLuCjISpua/5g0tLHt6YDe+VbLmLrxYtxwCXFb1UfMPu/7z/Q2ZkZ83usWzUXz3kLhISoQ9nNoarJbztG65gXKN3UCQbgHcKCZ7Fzsno6DF4GkTWLF0sFuQFtDYnO7EHz2gjde4JnNHkSP+5etuLa4wR3aQtazDOI6XXihdbCfclNqJlNSjbm7K2tWHKxHvCpkIYFxBFznvZKwuGWTqv9jp+YP0SavG254QDtMG8oqsQuOIMgPI4IHr8/A3fgEcG8KYYkQTvNjvScZFwhAKLZf9ikdD0jRStywax27AN3MFHX3K3ip2Uuv7C1oroj623Y+8L7tBOHwDigvamOvmExr4pVd9G1oSWvSp8B07xRElpfJM6PQ+s2QCx5GfbiNB9+M9pATBfTV79kApkyFuzGAW13GQDAyyXEAg0Ph6v4L1OwdAlM9v2U9QL4YDK8BWuNcQWQGUO58hMKX544RW/A6nnx1Rv2VnkiuCBZYCsOE63/LYUG65NFC+zs/PVkCRYiIZk0TOmG9FT03SnBVQQGqnYAoKjDo1jcopuh7fT5grwGoAAcN7oxWvVc2QCmPJcXtlHuCdfm4ienV6Qb9cGOg5Gglzf/w0IlqYpdelYQPNAAlFNGM6nlbiATwkNbAi4EIoG9CmvBNe5hQRrULSVtJUHJoqsoXiIpaLw5ZGHl0fk/S+/wtiD/jBz4ObpPEvPqj1pRJv40xtRmu3+1j1sOdLtnk5TPIcUXzvm5Nc+0fvfkRJJBGPKyP1fMPDcsI4BNQkNRNpIxUoWj4WwDAqBHzBq5MsTSmCF6Q279i8Fuu6QfEbujjwdh9fCScFkiqvUtII1azc92/CJ6GYrgOJVbveGBFOcJvsEkv1CF7Leukw1aXi9BU6EL7KBjgBJAavDV8/H5Cw022Yfl5NoDH70nYQBc/+s84UNwIIfdCq+i9jv6GZAlICj4KQJQASPCTm4X5+HdBsH6E7gAgoFJIjLX2pkye5gkoSWFe4FMD/7fDfsGdg+MRDkxNR/GYmxCy+ghF8CcWH9dsDzjrqAAtx0aMIb8iw7e2rrWDu0Hotp18fJfXcmdRulLRGyH1LqAXGGoo2E6clySpJIZjiCtEBCzHfC7gFj3iM3JuTJPN6nTXdBxwuAxHEnxH3jaE0MXTwSYDLcF/B4NXZepViDizWwMlVAS5IS2Fy5xdPBE66Qi3GpjsadOrKfH3E0bXAc6Mmzm+mJoJKtybEe8eB1stNGB7zmk8M++7DnhNjVPzTdqS30OJzO8BGj/o9QXcmS4QKO0tDodDQ+JmQ7JWVdB/6jw+BGpQE13/8GWGnXnY/sKQ5NJ/tTQMgHC+Toi6JA6hwXEavBVGpuDfV84HfKsNJZa9O3bZ9qh2jxdJEYwscjCLQBtNlRsNnXkmbBOMXycHgOROMNgmTpfipGRewwZwCc1R6XEo2/X+0vgsVLBv2Jt18iap1Pdp7j3cWdTGHgUkEAEod9F01onpwA88eHVJQSv3kiNsC656uwpwdLvFsxovE31cLr+UoupvVc+Kr+A1BLAwQUAAAACAAAACFcxG06iUMFAACcDAAAGQAAAGN1dHByZWFjaGVyL3RyYW5zY3JpYmUucHmNV91u2zYUvvdTHOjGUiGzTjsUmwsFaJOuGOClxZKhF4Eh0BJls5FIgaTsFlkeZtjFHiQvtkNSP5btAjOCWOIhv/P/HbpQsoI0LRrTKJamwKtaKgNUCGmo4VLoyaRdy8z3munu7auWYlLY0zU125Kvu6Of8dULcD8Xm279ipYlXZds4oWkkjkrdSe9ZZuKCRPDnaJCZ4rX+PxFqnwymeSsgDRrcprSHeUOJIxgdglrKcvFBPBj1Hf/YD+9vRaqpIa9mvQyxdBRcSgjG2Y8es52PGNpJhthUMElzN0x9i1jtYEP7gsjsjgG+5WWmnV21oqVkuYeEcOivak3UjB/MAiCA+0gGMs1ZM31zQ38Ai/x6f3y3S1cvAIpwGwZWDSmXJTfQotuBRX4IOMWseM5p1DzGvZb5qJaABfaYMhZHju1mouMAYWKa22z4jXSNUZKO5BayYxp7c4xVCELUJTbvQRNdhiFVIBpa0qGuyAMvF6SNeuS6iCGYSEXIoiGQI3SYz/1wwYSrDufqjQNPWzsfCq5Nsl9kKbW5zQNVlF/ts3Fb+7YB6WkGuNmUhguGtYvWpMxC2nOlbPZVmdYRxjoAJcDJ6+tBC0incZoDNpi2F0a1bI8bBHJppTr0AK9IFq+CKKjg2dd70117USurpfLUBtlMaPYhpclreiPu+V1+nH56f27ZXQC0Ubi0+2ZKPRRplp3dakaEVYMs+N8XLg2jaGkYtPQDVsAmhCD74D2xTVoKmjVLZRys+jb+P4e11axK+yVK/Ghc701rjwLitWk0v2W6xqruG3NL/71d6vB9yZih0VwRZViGypy2W2B6eNgx9MUwsfpx89/Tm2Be2MhSWBqm20KWPgMplcofooIIYEPmjuOxXaoMxwwO6cT/xVjCVV1Y1hqc5AEBbabuXgTjBUGVmHgFQZcmJ9bXdqzmE45Oh1jwRQSNTtlxLThWbOwT5bN+5CUKO4FXV6S7mEQ7ZEUU8Mrhu1d1Tq5U82BdIfUU/AS1R8JsDNybukrxT+kkR2XjU4N+2YSx19+59iNBdhWvG+5eYWu3K98tjCrqWIulwnMybynBzzq+uQwEIuR7drCjMrV0jy2AFUmUci9ebgn7i2G19gRTOT9Mj77RWf3nlg83Is1F0bRSc/uXcejKcTrxaX71XgbZnWM0ktXQ5JaXwitazQgbMMxstgqOWuzFYyttiv2odMY+6gk7v+BF47CC0nyRrlBjDM5hxYOZqMUXCY4L+ZjDhinqD033uFazjKUL0wj4bHDfwlv5gsyL56w6OFxbMcgq7hoC7+dhAMFhH0Fu8N9GUMHk4xA4z7KSfcQtcx13DcnJOZWM5pt2cnqiN4wDEFtgnhghYHc4C9HZLjFfsUdJf2A7nBbrbDt48lZ4sPMDeYQrm1D4o1lyI+NfNCdev73+R8JX5//RkrHbsMQKUZxGO8YN5YJOx47G2ZiSRZHUWZCeyEj9nKgwwPtiJW7Lg+ZyGSO0zwJGlPMkLGw1tq7U4fWut85cXLnGjw4d8358bQfaTiaRMMMiltaPZw8tmeDkqoNm+1ez9D5tQzcIDq5EvTXM6Dari3O1DoS3ZYCldBoqvAuhJMEBwpuforedlcHN3wEBRwjXeR5cegB1weXuT6bt3gfQ7gFoppRYmv0BUq2Q4Wv5rOf5rZpcDArCk1F7X1uQ/1G3HYBW6noYcL/f+Tq5jRwusLq7eJ11CYEbcA2I9UDXmRC/9IOE1+IqXxwr9HxwT1SBfM15Woub3AMhYOlxEhfkY4Itf1dQXXGuZ8zbvGoEg8pZMCZ/AdQSwMEFAAAAAgAAAAhXB7zjNT/CQAAjhgAABQAAABjdXRwcmVhY2hlci92aWRlby5web1Y647bxhX+r6eYMghA2hS9knfdRCmDOgsnNRB7jbWRohAEYiQOJXYpkuWQumS7T9PfeQq/WL9zZniRdl34VxdYkXM71+9chklVbEUUJU3dVCqKRLoti6oWMs+LWtZpkevRyM79Uxd5+643TZ1m3Yi26jpd6W6mWZZVsVJajxLiEMtarjKptdIti27K7ChlvcnSZbv6AcPRaMQbxG9prIo3VVVU7m2T1+lW8cCbjQT+SqIxGv21Jzg49TZPCrNtn8b1ZibSvObhRqXrTd2P46ZifWciyQpp90gdySZOi5lYFkUGJrFKRNXkUZJsS7V2ZbXWM5FB97muq4UvVvt4xrKLf4v3Ra5EyA9PjH/kFysxTIOV3kgBaLq8RH9zx5B3fOGMj/y7gS7REl5RFY+zYp2pncpooMgWeHlG0iz8jgxkCfE/mJAle7lo6rKpw09Vo/rFWh2GUx7/pgnLGlQKB/NVESvxp1BczLpTlUy1GvonsbKLRGabopmJeyag6xhi4lGlpevNx9OLi4vZ4sHxrE2xaalcwoAxHxvszIFfYTUmwwbaDS3jjMsKfo6SotrKmiYIy7ygN8U+glhKbnU/0W3ECkvlLf7fZnz/+T+FSIpUlIXWn/+Ar0WmKlGI3ec/sA12hVgPT9uX7MqYRkTAYqRsAFDH2m03Q3azZUc8sSeH3K6rwbESCNGcz86d1jIL0kHPHZJ9FdXHUmEqDIXD5x3PNzBvtTVUUz3A/BfUvBGy+leT7lqFRE5612oroEq2kQK2MhoHVimOY9/GLySHY13mN3d4yVlAmsGk2YhZPl3ZtIaDcqndbl+wVrXr1HJNKLh/8MyYdxOgLjwBw1x4hggbCafIToPTNBWR3SLKCDg1X3gD5R9zpgMDRiljsn094QibdufB1P3+whfTP18MyJ+bxbz4Zt6ozujro8o9PRsaCt2kIRBaOt10mydDTpOugYmNl4VRpt1CdgugwYBmm1BDmR/dp/DEq84XYei1YWXSBkBbyVVtaBpHmvzhCyB8kEroZdZaEqWNloNUR0maKXfopD63z51xajMAE/Y4rZisIVf0mJh3zjCTV8hoPF7NJD3L1TbSk1eZsjQo4loMGkdgxqoRIZXn7sGWHhYY8JgNN2/lwZ0aXFdFk8fuQbwQU88Tz/BryayqooyWxcHV1Srac2kDb7xv7Ds4dvP03s6vVF6rKmoFeKJ21U2ZqTlvPvlZGCEdx3H3AJ8vDr44ArmJqDdKZKhISteWL3ME1vK4oL1Flf5e5LXMsiMLUKlYAN2tLGK/UblYp7BMAPLMpiZ6BG4mCAMwydatrDUmWWPxo93d+3ZDxYPXxvb5rZj2MYDVbZq7xhcb2NWc93xLeGyf35LBu1pwsESJ8ZSk6OS32U+oDDmvne0OHmzmIpbkWxAaCyZCyabnufd6ZhYK1tBYPJAw2D9qFXgk6ahVvFeNmJyodmIPw+2IE267hgh68cKSOhEBOeiI9SMzMhCMVa0Qj7pU8o70PY1J9IlVbUHmC5XH3buWWwBMMx7Be3LFqBuisQPaOxWnMh/gx6LnHHSJXGFuh5pJs6ssLQmxQhdNhYUyPcAxHbJs17naQZFBlkC+kjWCF/MI6WupVzJW19RdpkmKVsybCfENjLIFn1hwH0vMthKMtkXcZFwibkqVX/8mrs4dSWrx3MoQhuJgFDxiQ+wDSoTBRsrKbtbiuXAG4wjcyRqkdgRXyCarg8M2s1UTTYslz/n/2vQwbp/dvKHe2I3sSHKr+CQ9ngvOsNagPF/whC4lFThCNNxLwCKX4zkJLkw1MIFcHXuilOtTslMl8zVEMlAYcOUThG6m9RxkrvDLrJ7h5AtmZ48RK19MvJPDpI5GZWLrvv4Qfbi9wc/Nx+jdxzfXPmg/ExOk79NDxZ0Pj8otuwUEUIFgi5MtbS25O5WVWcIZad6okwU4KiNy37+6gNRMPNAbWar5hBuss5kfeSPnD5jvlNIWedM6tFI6/V25fNj0YRD8EDIzvB3bt5R8VRaZKd908u37T29uo9e3b157nEBZPHSm4GbYMs0TxutKHi3f1a6+LjJ0cSyMb7B78+vNbfTTL7fTX25f/+PUWoRMzcZkwAYmV7wDTtOPxNkl2r6R4mdU9aIKJwGcidT1npqQZVHp8IrHH6Fw6E4v0QRdeo98kiHLMTPvsV9QnyKf8ldkYcobfXGnjmEmt8tYigRZaT5dABTJ/OXCe+xaA/pAloiP2EUlfm4yt0fFh6S3XWKaU3HrZTAoypTUyu3CjYS1FD3xl7bOt2hG4r36H/Fnx/39O9hyduwo2rS8bNIspmYHk+g8caPq28AvtwXk/TRfz6h/+VKT4AtkqSgHTOy2BMDX5r7MORyTRn66kocicfAM79tDDw6M5zozPhWnVcgvDgcDvRkYOk5nLisT94pIgkkBKBXOIwu5p8hz5hez3UKXWVrP5WK+XPzgnG+QC3ZdeM/WeJjxExcsZCckVBSbNbkzkihtKHDc4IZpvqrImz51XucnfXRiy6ypwunVbDpfrp/guTzjOZ7Okyf3rRe0QMUsk8fQ/ft4772Yzty/jTd4+khtWiJYfDLrw3y36AnYK1PXmlH4tV0i4SCw1wZ+b68MLI2Fgj9oaE6tXxW6RqfODuovf9b+1uLGLmQTWPL+gP/jg/+knZ9Swn5uQZCBPxVwt7+y2pbCwOIM0KYSDXsNnhn0Gy0gI7quG+DyFMnSf4Qwc/uiuosAzeHcIGK68WYwPomcQaV8FD32OgmkGxbYf9p9+6Oz+8s3ov3AwtHM3800LtFoNGHYEm2I3N/tZRVnlEsLaz6IxhfWVhm04fUGa1QMdkpQKOrgRN1ge4dftwRlSGc+aeC+hTQTFXc89Dor5qZKJs59a8BA4wr/EGDRYNHtGL/oTnjBvkprxT5wW2dQW4hLIYwXOk2djL+zoU83R5MRQvGzBOa6hNAaT0g0HGQnt5vy6IpHWgxSqPlyGayK8oimUfWbfTGQ0TF5yBcY6ugJtc9lohXTOqKKldRxP8q554FlQdJHWJ9L/Z62tXK15i7LGWuSCpY2+A5eJg986zy5qlJLUGQ7VBm+tNbmxKAh688Z+aJVQfXm4Cxas5qE0H3/7HQ22j1H/P+ACJeLrGjiHLf/8G04nlzOPn3AI7ia/Xr7OpxMfIDHVLLw8js0WEiyfWpilUBoziRJlq0sSSaK/eFQLob3amnu3uOlGeLefWelpkw0+zry5kC9LekjHuw68HyH4LxtfTpKHW0Shj8wZunyMH11aT8xKuQweqUa3GyN0FVCz6n5NIBLR5Rsec+x2V1OL1igMX9AeIktAwa4USSZ+RrlPE+krtlvGFmhWTxzwChz9lWaP0WHrV7eSXYLTFB/bYQDDS3TDlfUJXbkutkno4w/sVgCfnfo/EsIT47+C1BLAQIUAxQAAAAIAAAAIVyZcbCpbQAAAG0AAAAXAAAAAAAAAAAAAACAAQAAAABjdXRwcmVhY2hlci9fX2luaXRfXy5weVBLAQIUAxQAAAAIAAAAIVwAV15OVgIAADkFAAAXAAAAAAAAAAAAAACAAaIAAABjdXRwcmVhY2hlci9fX21haW5fXy5weVBLAQIUAxQAAAAIAAAAIVzgGukdNwUAAHkMAAAXAAAAAAAAAAAAAACAAS0DAABjdXRwcmVhY2hlci9jYXB0aW9ucy5weVBLAQIUAxQAAAAIAAAAIVyqVtTlRAEAAHoCAAAVAAAAAAAAAAAAAACAAZkIAABjdXRwcmVhY2hlci9jb25maWcucHlQSwECFAMUAAAACAAAACFcDRftDFgEAAAnCQAAFwAAAAAAAAAAAAAAgAEQCgAAY3V0cHJlYWNoZXIvZG93bmxvYWQucHlQSwECFAMUAAAACAAAACFcoG9kkRkMAABEHwAAGQAAAAAAAAAAAAAAgAGdDgAAY3V0cHJlYWNoZXIvaGlnaGxpZ2h0cy5weVBLAQIUAxQAAAAIAAAAIVzXtztbuAEAALUEAAAVAAAAAAAAAAAAAACAAe0aAABjdXRwcmVhY2hlci9tb2RlbHMucHlQSwECFAMUAAAACAAAACFcXYW2c6AGAACWDwAAFwAAAAAAAAAAAAAAgAHYHAAAY3V0cHJlYWNoZXIvcGlwZWxpbmUucHlQSwECFAMUAAAACAAAACFcxG06iUMFAACcDAAAGQAAAAAAAAAAAAAAgAGtIwAAY3V0cHJlYWNoZXIvdHJhbnNjcmliZS5weVBLAQIUAxQAAAAIAAAAIVwe84zU/wkAAI4YAAAUAAAAAAAAAAAAAACAAScpAABjdXRwcmVhY2hlci92aWRlby5weVBLBQYAAAAACgAKAK8CAABYMwAAAAA="
APP_DIR = Path("/content/cutpreacher_app")
zipfile.ZipFile(io.BytesIO(base64.b64decode(PACKAGE_ZIP))).extractall(APP_DIR)
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
for name in [m for m in sys.modules if m == "cutpreacher" or m.startswith("cutpreacher.")]:
    del sys.modules[name]

from google.colab import drive, userdata
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/CutPreacher")
for sub in ("entrada", "saida", "fontes"):
    (BASE / sub).mkdir(parents=True, exist_ok=True)

FONT_NAME = "Montserrat ExtraBold"
font_file = BASE / "fontes" / "Montserrat-ExtraBold.ttf"
if not font_file.is_file():
    try:
        urllib.request.urlretrieve(
            "https://github.com/JulietaUla/Montserrat/raw/master/fonts/ttf/Montserrat-ExtraBold.ttf", font_file
        )
    except Exception as exc:
        print(f"Não consegui baixar a fonte Montserrat ({exc}); a legenda usará a fonte padrão.")
        FONT_NAME = "DejaVu Sans"

try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✔ Chave da API encontrada.")
except Exception:
    print("⚠️ Chave ANTHROPIC_API_KEY não encontrada nos Secrets (ícone 🔑 à esquerda). Veja as instruções no topo.")

import ctranslate2
if ctranslate2.get_cuda_device_count() > 0:
    print("✔ GPU ativa — a transcrição será rápida.")
else:
    print("⚠️ Sem GPU: vai funcionar, mas a transcrição demora mais. Ative em Ambiente de execução → Alterar o tipo → GPU T4.")

import cutpreacher
print(f"✔ CutPreacher {cutpreacher.__version__} pronto. Vídeos serão salvos em: {BASE / 'saida'}")


In [ ]:
#@title 2️⃣ Gerar os cortes (preencha e clique no ▶)
link_do_youtube = ""  #@param {type:"string"}
#@markdown Ou, se o YouTube bloquear: nome do arquivo enviado para a pasta CutPreacher/entrada do Drive
arquivo_no_drive = ""  #@param {type:"string"}
quantidade_de_cortes = 3  #@param {type:"slider", min:3, max:8, step:1}
duracao_maxima_segundos = 59  #@param {type:"slider", min:30, max:60, step:1}
enquadramento = "rosto"  #@param ["rosto", "centro", "desfocado"]
#@markdown **rosto**: segue o pregador · **centro**: corte no meio da imagem · **desfocado**: imagem inteira com fundo desfocado

from cutpreacher.config import Settings
from cutpreacher.pipeline import run

if arquivo_no_drive.strip():
    fonte = str(BASE / "entrada" / arquivo_no_drive.strip())
elif link_do_youtube.strip():
    fonte = link_do_youtube.strip()
else:
    raise ValueError("Preencha o link do YouTube (ou o nome do arquivo no Drive).")

config = Settings(
    num_clips=int(quantidade_de_cortes),
    max_clip_seconds=float(duracao_maxima_segundos),
    framing=enquadramento,
    font_name=FONT_NAME,
    fonts_dir=str(BASE / "fontes"),
    cookies_file=str(BASE / "cookies.txt"),
)
resultado = run(fonte, BASE / "saida", config, work_root=Path("/content/cutpreacher_work"))

print()
for arquivo, corte in zip(resultado.files, resultado.clips):
    print(f"🎬 {arquivo.name}\n   {corte.title}\n   {corte.description}\n   {' '.join(corte.hashtags)}\n")
print(f"📁 Abra o Google Drive → CutPreacher/saida/{resultado.output_dir.name}")


In [ ]:
#@title 3️⃣ (Opcional) Baixar os vídeos direto neste aparelho
from google.colab import files
for arquivo in resultado.files:
    files.download(str(arquivo))
